In [9]:
# Initialize Otter
import otter
grader = otter.Notebook("lab.ipynb")

# Lab 3 – Merging and Pivoting

## DSC 80, Fall 2026

### Due Date: Wednesday, October 21, 2026 at 11:59 p.m. Pacific

## Instructions

Welcome to the third DSC 80 lab this quarter!

Much like in DSC 10, this Jupyter Notebook contains the statements of the problems and provides code and Markdown cells to display your answers to the problems. Unlike DSC 10, the notebook is *only* for displaying a readable version of your final answers. The coding will be done in an accompanying `lab.py` file that is imported into the current notebook, and **you will only submit that `lab.py` file**, not this notebook!

Some additional guidelines:
- **Unlike in DSC 10, labs will have both public tests and hidden tests.** The bulk of your grade will come from your scores on hidden tests, which you will only see on Gradescope after the assignment deadline.
- **Do not change the function names in the `lab.py` file!** The functions in the `lab.py` file are how your assignment is graded, and they are graded by their name. If you changed something you weren't supposed to, you can find the original code in the [course GitHub repository](https://github.com/dsc-courses/dsc80-2026-fa).
- Notebooks are nice for testing and experimenting with different implementations before designing your function in your `lab.py` file. You can write code here, but make sure that all of your real work is in the `lab.py` file, since that's all you're submitting.
- You are encouraged to write your own additional helper functions to solve the lab, as long as they also end up in `lab.py`.

**To ensure that all of the work you want to submit is in `lab.py`, we've included a script named `lab-validation.py` in the lab folder. You shouldn't edit it, but instead, you should call it from the command line (e.g. the Terminal) to test your work.** More details on its usage are given at the bottom of this notebook.

**Importing code from `lab.py`**:

* Below, we import the `.py` file that's contained in the same directory as this notebook.
* We use the `autoreload` notebook extension to make changes to our `lab.py` file immediately available in our notebook. Without this extension, we would need to restart the notebook kernel to see any changes to `lab.py` in the notebook.
    - `autoreload` is necessary because, upon import, `lab.py` is compiled to bytecode (in the directory `__pycache__`). Subsequent imports of `lab` merely import the existing compiled python.

In [10]:
%load_ext autoreload
%autoreload 2

In [11]:
from lab import *

In [12]:
import os
import io
from pathlib import Path
import pandas as pd
import numpy as np

<div class="alert alert-block alert-danger" markdown="1">

**There are only two functions in this lab in which you may use a `for`-loop:**
- **`read_linkedin_survey` in Question 1.**
- **`read_student_surveys` in Question 2.**
    
**You may lose points if you use a `for`-loop or `while`-loop in any other question!**

</div>

## Part 1: Combining Data

### Question 1 – Making Connections 🤝

A group of students decided to send out a survey to their connections on LinkedIn. Each student asks 1000 of their connections for their first and last name, the company they currently work at, their job title, their email, and the university they attended.

**Your job is to combine all the data contained in the files `survey*.csv` (stored within the `data/responses` folder) into a single DataFrame. The number of files and the number of rows in each file may vary, so don't hardcode your answers!** To do so, implement the following two functions.

#### `read_linkedin_survey`

Complete the implementation of the function `read_linkedin_survey`, which takes in a Path for the directory where `survey*.csv` files are located and outputs a single DataFrame with six columns titled `'first name'`, `'last name'`, `'current company'`, `'job title'`, `'email'`, and `'university'` (in that order) containing the all of survey information from all the `.csv` files. Make sure to reset the index of the combined DataFrame before returning it so that the index is unique. When the function takes in an invalide directory it should raise a `FileNotFoundError`.

***Hints***:

- Take a look at a few of the files in the `responses` folder. You may have to do some data cleaning to combine the DataFrames!

- You can list the files in a directory using `.iterdir()`.

***Note***: Remember to use `Path()` to build file paths. Calling `.iterdir()` on a `Path()` object will automatically throw a `FileNotFoundError`.

<br>

#### `com_stats`

Complete the implementation of the function `com_stats`, which takes in a DataFrame returned by `read_linkedin_survey` and returns a list containing, in the following order: 
- The proportion of people who went to a university with the string `'Ohio'` in its name that have the string `'Programmer'` somewhere in their job title.
- The number of job titles that **end** with the exact string `'Engineer'`. Note that we're asking for the number of job titles, **not** the number of people!
- The job title that has the longest name (there are no ties).
- The number of people who have the word `'manager'` in their job title, uppercase or lowercase (`'Manager'`, `'manager'`, and `'mANAgeR'` should all count).

In [13]:
def read_linkedin_survey(dirname):

    df = pd.DataFrame()

    path = Path(dirname)

    for child in path.iterdir():
        try:
            child_df = pd.read_csv(child)
            child_df.columns = [colname.lower().strip().replace('_', '').replace(' ', '') for colname in child_df.columns]
            df = pd.concat([df, child_df], axis=0, ignore_index=True).fillna('')
        except FileNotFoundError:
            raise(FileNotFoundError)

    df = df.rename(columns={
        'firstname' : 'first name',
        'lastname' : 'last name',
        'currentcompany' : 'current company',
        'jobtitle' : 'job title',
        'email' : 'email',
        'university' : 'university'
    })

    return df.reset_index()[['first name', 'last name', 'current company', 'job title', 'email', 'university']]


def com_stats(df):
    results = []

    copy = df.copy()

    copy['title length'] = copy['job title'].apply(lambda x: len(x))
    
    results.append(
        df[df['university'].str.contains('Ohio')]['job title'].str.contains('Programmer').mean()
    )
    results.append(
        len(list(filter(lambda x: x[-8:] == 'Engineer', df['job title'].unique())))
    )
    results.append(
        copy.sort_values(by='title length', ascending=False)['job title'].head(1).iloc[0]
    )
    results.append(
        df['job title'].str.lower().str.contains('manager').sum()
    )

    return results

In [14]:
df = pd.read_csv('data/responses/survey1.csv')
a = [colname.lower().strip().replace('_', '').replace(' ', '') for colname in df.columns]
a

['firstname', 'lastname', 'jobtitle', 'email', 'currentcompany', 'university']

In [15]:
dirname = Path('data') / 'responses'
df = read_linkedin_survey(dirname)

df

,first name,last name,current company,job title,email,university
0,Marsh,Kyllford,"Abernathy, Brown and Stiedemann",Financial Analyst,mkyllford0@naver.com,Ganja State University
1,Igor,Greatex,"Stiedemann, Eichmann and Will",Data Coordiator,igreatex1@phpbb.com,Meikai University
2,Garrick,Truscott,Collins-Hintz,Tax Accountant,gtruscott2@weather.com,Saint Paul University
3,Eleanore,Sansam,Trantow and Sons,Senior Quality Engineer,esansam3@pbs.org,Universidad del Rosario
4,Laurent,Bagley,Weissnat and Sons,Financial Advisor,lbagley4@statcounter.com,St. Petersburg StateMechnikov Medical Academy
...,...,...,...,...,...,...
4995,Hurleigh,Sphinxe,Kozey-Hamill,Marketing Manager,hsphinxern@eventbrite.com,Southampton Solent University
4996,Michele,Finch,Ernser and Sons,Account Coordinator,mfinchro@jigsy.com,Glion Institute of Higher Education
4997,Harlie,Heister,Schuppe-McClure,Technical Writer,hheisterrp@last.fm,Tianjin University of Finance & Economics
4998,Phillipe,Dibble,"Barrows, Batz and Dickens",Analog Circuit Design manager,pdibblerq@jugem.jp,University of South Bohemia


In [16]:
df[df['university'].str.contains('Ohio')]['job title'].str.contains('Programmer').mean()

np.float64(0.15384615384615385)

In [17]:
len(list(filter(lambda x: x[-8:] == 'Engineer', df['job title'].unique())))

13

In [18]:
max(list(map(lambda x: len(x), df['job title'].unique())))

36

In [19]:
copy = df.copy()
copy['title length'] = copy['job title'].apply(lambda x: len(x))
copy.sort_values(by='title length', ascending=False)['job title'].head(1).iloc[0]

'Business Systems Development Analyst'

In [20]:
df['job title'].str.lower().str.contains('manager').sum()

np.int64(369)

In [21]:
# do not edit this cell -- it is needed for the tests
dirname = Path('data') / 'responses'
q1_out = read_linkedin_survey(dirname)
stats_out = com_stats(q1_out)

In [22]:
q1_out

,first name,last name,current company,job title,email,university
0,Marsh,Kyllford,"Abernathy, Brown and Stiedemann",Financial Analyst,mkyllford0@naver.com,Ganja State University
1,Igor,Greatex,"Stiedemann, Eichmann and Will",Data Coordiator,igreatex1@phpbb.com,Meikai University
2,Garrick,Truscott,Collins-Hintz,Tax Accountant,gtruscott2@weather.com,Saint Paul University
3,Eleanore,Sansam,Trantow and Sons,Senior Quality Engineer,esansam3@pbs.org,Universidad del Rosario
4,Laurent,Bagley,Weissnat and Sons,Financial Advisor,lbagley4@statcounter.com,St. Petersburg StateMechnikov Medical Academy
...,...,...,...,...,...,...
4995,Hurleigh,Sphinxe,Kozey-Hamill,Marketing Manager,hsphinxern@eventbrite.com,Southampton Solent University
4996,Michele,Finch,Ernser and Sons,Account Coordinator,mfinchro@jigsy.com,Glion Institute of Higher Education
4997,Harlie,Heister,Schuppe-McClure,Technical Writer,hheisterrp@last.fm,Tianjin University of Finance & Economics
4998,Phillipe,Dibble,"Barrows, Batz and Dickens",Analog Circuit Design manager,pdibblerq@jugem.jp,University of South Bohemia


In [23]:
grader.check("q1")

q1 results: All test cases passed!

### Question 2 – Survey Says... 👨‍👩‍👧‍👦

Professor Aritra often sends out extra credit surveys asking students for their favorite animals, movies, and other favorite things. These surveys are stored in the `data/extra-credit-surveys` folder. Each file in that folder corresponds to a different survey question (except for `favorite1.csv`, which contains students' names and IDs).

Here's how extra credit works:
- Each student who has completed at least 50% of the survey questions receives 5 points of extra credit.
- If there is at least one survey question that at least 90% of the class answered (e.g. favorite animal), **everyone** in the class receives 1 point of extra credit. This overall class extra credit only applies twice, so if for example 95% of students answer the favorite color survey question and 91% answer the favorite animal survey question, and and 97% answer the favorite movie question, the entire class still receives 2 extra point as a class, not 3.
- Note that this means that the most extra credit any student can earn is 7 points.

#### `read_student_surveys`

Complete the implementation of the function `read_student_surveys` which takes in a Path describing the directory to a folder containing `favorite*.csv` files and outputs a DataFrame containing all of the survey data combined, indexed by student ID (a value 1-1000).

***Note***: Remember to use `Path()` to build file paths.


<br>

#### `check_credit`

Complete the implementation of the function `check_credit` which takes in a DataFrame returned by `read_student_surveys` and outputs a DataFrame indexed by student ID (a value 1-1000) with two columns:
- `'name'`, containing the name of each student, and
- `'ec'`, containing the number of extra credit points each student earned.

***Note***: For the genres survey, `'(no genres listed)'` does not count as a valid response for receiving extra credit. 

In [113]:
def read_student_surveys(dirname):
    df = pd.DataFrame({'id': np.arange(1, 1001)})

    path = Path(dirname)

    for child in path.iterdir():
        try:
            child_df = pd.read_csv(child)
            df = pd.merge(df, child_df, on='id')
        except FileNotFoundError:
            raise(FileNotFoundError)

    return df.set_index('id')


def check_credit(df):
    r = df[['name']].copy()

    valid = (df.notnull() & (df != '(no genres listed)'))

    r['ec'] = 5*(df[valid].notnull().drop(columns='name').sum(axis=1) >= 0.5*valid.shape[1]) + min(2, sum((df[valid].notnull().drop(columns='name').sum(axis=0)/valid.shape[0]) >= 0.90))

    return r

In [114]:
survey6 = pd.read_csv('data/extra-credit-surveys/favorite6.csv')
survey5 = pd.read_csv('data/extra-credit-surveys/favorite5.csv')

pd.merge(survey5, survey6, on='id')

,id,plant,color
0,1,NaN,Red
1,2,NaN,Khaki
2,3,NaN,Red
3,4,NaN,Yellow
4,5,NaN,Fuscia
...,...,...,...
995,996,NaN,Purple
996,997,NaN,Blue
997,998,NaN,NaN
998,999,NaN,Maroon


In [115]:
dirname = Path('data') / 'extra-credit-surveys'
df = read_student_surveys(dirname)
df

,name,movie,genre,animal,plant,color
id,,,,,,
1,Myrtia,NaN,(no genres listed),Long-crested hawk eagle,NaN,Red
2,Nathanil,NaN,Documentary,Euro wallaby,NaN,Khaki
3,Joni,"Glass-blower's Children, The (Glasblåsarns barn)",NaN,Brown brocket,NaN,Red
4,Prentice,NaN,(no genres listed),"Peccary, white-lipped",NaN,Yellow
5,Claudette,NaN,NaN,"Capuchin, brown",NaN,Fuscia
...,...,...,...,...,...,...
996,Addie,Kung Phooey!,Horror|Mystery|Sci-Fi,"Eland, common",NaN,Purple
997,Valaria,Angel Heart,NaN,Agouti,NaN,Blue
998,Gunilla,NaN,NaN,"Shelduck, european",NaN,NaN


In [116]:
valid = (df.notnull() & (df != '(no genres listed)'))

5*(df[valid].notnull().drop(columns='name').sum(axis=1) >= 0.5*valid.shape[1])

id
1       0
2       5
3       5
4       0
5       0
       ..
996     5
997     5
998     0
999     0
1000    5
Length: 1000, dtype: int64

In [122]:
min(2, sum((df[valid].notnull().drop(columns='name').sum(axis=0)/valid.shape[0]) >= 0.90))

1

In [123]:
r = df[['name']].copy()

valid = (df.notnull() & (df != '(no genres listed)'))
r['ec'] = 5*(df[valid].notnull().drop(columns='name').sum(axis=1) >= 0.5*valid.shape[1]) + min(2, sum((df[valid].notnull().drop(columns='name').sum(axis=0)/valid.shape[0]) >= 0.90))

r

,name,ec
id,,
1,Myrtia,1
2,Nathanil,6
3,Joni,6
4,Prentice,1
5,Claudette,1
...,...,...
996,Addie,6
997,Valaria,6
998,Gunilla,1


In [119]:
# do not edit this cell -- it is needed for the tests
dirname = Path('data') / 'extra-credit-surveys'
q2_out = read_student_surveys(dirname)
check_credit_out = check_credit(q2_out)

In [120]:
check_credit_out

,name,ec
id,,
1,Myrtia,1
2,Nathanil,6
3,Joni,6
4,Prentice,1
5,Claudette,1
...,...,...
996,Addie,6
997,Valaria,6
998,Gunilla,1


In [121]:
grader.check("q2")

q2 results: All test cases passed!

### Question 3 – Paw Patrol 🐾

You are analyzing data from a veterinarian clinic. The datasets contain several types of information from the clinic, including its customers (pet owners), pets, available procedures, and procedure history. The column names are self-explanatory. These DataFrames are provided to you:
-  `owners` stores the customer information, where every `'OwnerID'` is unique (verify this yourself).
-  `pets` stores the pet information. Each pet belongs to a customer in `owners`.
-  `procedure_detail` contains a catalog of procedures that are offered by the clinic.
-  `procedure_history` has procedure records. Most procedures were given to a pet in `pets`.

Complete the implementation of the following three functions, which each ask you to answer a specific question.

#### `most_popular_procedure`

What is the most popular `'ProcedureType'` amongst all pets in the `pets` DataFrame? Complete the implementation of the function `most_popular_procedure`, which takes in two DataFrames, `pets` and `procedure_history`, and returns the name of the most popular `'ProcedureType'` as a string.

Note that some pets are registered but haven't had any procedures performed. Also, some pets that have had procedures done are not registered in `pets`.

<br>

#### `pet_name_by_owner`

What is the name of each customer's pet(s)? Complete the implementation of the function `pet_name_by_owner`, which takes in two DataFrames, `owners` and `pets`, and returns a Series whose index contains owner first names, and whose values are pet names as **strings**. If an owner has multiple pets, the value corresponding to that owner should instead be a **list of pet names as strings**.

Note that owner first names are not necessarily unique, and so the Series you return will not necessarily have a unique index.

<br>

#### `total_cost_per_city`

Note that the `owners` DataFrame has a `'City'` column, describing the city in which each pet owner and their pets live. How much did each city spend in total on procedures? Complete the implementation of the function `total_cost_per_city`, which takes in four DataFrames, `owners`, `pets`, `procedure_history`, and `procedure_detail`, and returns a Series indexed by `'City'` that describes the total amount that each city has spent on pets' procedures.

***Hint***: At some point, you may have to merge on multiple columns.

***Note***: Some owners may have never visited the veterinarian clinic in their city. This means some cities may have zero operational costs.

In [31]:
def most_popular_procedure(pets, procedure_history):
    df = pd.merge(pets, procedure_history, on='PetID')

    return df.groupby('ProcedureType').count().iloc[:,0].sort_values(ascending=False).index[0]

def pet_name_by_owner(owners, pets):
    df = pd.merge(owners, pets, on='OwnerID')

    df['Name_y'] = df['Name_y'].apply(lambda x : [x])

    df = df[['Name_x', 'Surname', 'Name_y']].groupby(['Name_x', 'Surname']).sum()
    df = df.reset_index().drop(columns='Surname').set_index('Name_x')['Name_y']

    return df.apply(lambda x : x[0] if len(x) == 1 else x)

def total_cost_per_city(owners, pets, procedure_history, procedure_detail):
    procedures = pd.merge(procedure_history, procedure_detail, on=['ProcedureSubCode', 'ProcedureType'])
    master = owners.merge(pets.merge(procedures, on='PetID', how='left'), on='OwnerID', how='left')

    master = master.rename(columns=
                  {
                      'Name_x' : 'Name_pet',
                      'Name_y' : 'Name_owner',
                  })

    return master.groupby('City').sum()['Price']

In [32]:
pets_fp = Path('data') / 'pets' / 'Pets.csv'
procedure_history_fp =  Path('data') / 'pets' / 'ProceduresHistory.csv'
owners_fp = Path('data') / 'pets' / 'Owners.csv'
procedure_detail_fp = Path('data') / 'pets' / 'ProceduresDetails.csv'
pets = pd.read_csv(pets_fp)
procedure_history = pd.read_csv(procedure_history_fp)
owners = pd.read_csv(owners_fp)
procedure_detail = pd.read_csv(procedure_detail_fp)

In [33]:
# do not edit this cell -- it is needed for the tests
pets_fp = Path('data') / 'pets' / 'Pets.csv'
procedure_history_fp =  Path('data') / 'pets' / 'ProceduresHistory.csv'
owners_fp = Path('data') / 'pets' / 'Owners.csv'
procedure_detail_fp = Path('data') / 'pets' / 'ProceduresDetails.csv'
pets = pd.read_csv(pets_fp)
procedure_history = pd.read_csv(procedure_history_fp)
owners = pd.read_csv(owners_fp)
procedure_detail = pd.read_csv(procedure_detail_fp)

out_01 = most_popular_procedure(pets, procedure_history)
out_02 = pet_name_by_owner(owners, pets)
out_03 = total_cost_per_city(owners, pets, procedure_history, procedure_detail)

In [34]:
grader.check("q3")

q3 results: All test cases passed!

## Part 2: Pivot Tables

### Question 4 – Summarizing Sales 💰

Recall from [Lecture 3](https://dsc80.com/resources/lectures/lec03/lec03-filled.html#Pivot-tables-using-the-pivot_table-method), a pivot table allows you to aggregate the entries in a DataFrame based on two categorical columns. In this question, you are given a simple dataset, `data/sales.csv`, and are asked to solve a few problems using the `pivot_table` method. 

**We have provided outlines for the DataFrames you need to create, but yours may have a different number of rows and columns and different values.**

***Note***: If it helps, [here](https://jakevdp.github.io/PythonDataScienceHandbook/03.09-pivot-tables.html) is another great resource that provides an overview of `pivot_table` with many examples from the Titanic dataset.


#### `average_seller`

Complete the implementation of the function `average_seller`, which takes in the `sales` DataFrame and returns a DataFrame that contains the average sales for each seller, indexed by `'Name'` and containing the column `'Average Sales'`. There should not be any `NaN`s.

***Note***: You may be able to implement `average_seller` without using `pivot_table`.

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Average Sales</th>
    </tr>
    <tr>
      <th>Name</th>
      <th></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>Jones</th>
      <td>0</td>
    </tr>
    <tr>
      <th>Smith</th>
      <td>0</td>
    </tr>
    <tr>
      <th>Trump</th>
      <td>0</td>
    </tr>
  </tbody>
</table>

<br>

#### `product_name`

Complete the implementation of the function `product_name` that takes in the `sales` DataFrame and returns a DataFrame that contains the total sales for each product, indexed by `'Name'`. Do not fill in `NaN`s.

<table border="1" class="dataframe">
  <thead>
    <tr>
      <th>Product</th>
      <th>boat</th>
      <th>book</th>
      <th>hotel</th>
    </tr>
    <tr>
      <th>Name</th>
      <th></th>
      <th></th>
      <th></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>Jones</th>
      <td>NaN</td>
      <td>0.0</td>
      <td>NaN</td>
    </tr>
    <tr>
      <th>Smith</th>
      <td>NaN</td>
      <td>0.0</td>
      <td>NaN</td>
    </tr>
    <tr>
      <th>Trump</th>
      <td>0.0</td>
      <td>0.0</td>
      <td>0.0</td>
    </tr>
  </tbody>
</table>

<br>

#### `count_product`

Complete the implementation of the function `count_product` that takes in the `sales` DataFrame and returns a DataFrame that contains the total number of items sold product-wise and name-wise per date. Replace `NaN`s with 0s. Don't reset the index after pivoting.

<table border="1" class="dataframe">
  <thead>
    <tr>
      <th></th>
      <th>Date</th>
      <th>01.01.2012</th>
      <th>02.20.2013</th>
      <th>02.25.2015</th>
    </tr>
    <tr>
      <th>Product</th>
      <th>Name</th>
      <th></th>
      <th></th>
      <th></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>boat</th>
      <th>Trump</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
    <tr>
      <th rowspan="3" valign="top">book</th>
      <th>Jones</th>
      <td>0</td>
      <td>1</td>
      <td>0</td>
    </tr>
    <tr>
      <th>Smith</th>
      <td>1</td>
      <td>0</td>
      <td>0</td>
    </tr>
    <tr>
      <th>Trump</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
    <tr>
      <th>hotel</th>
      <th>Trump</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
  </tbody>
</table>

<br>

#### `total_by_month`

Complete the implementation of the function `total_by_month` that takes in the `sales` DataFrame and returns a pivot table that contains the total sales name-wise, product-wise per month. Replace `NaN`s with 0s. Don't reset the index after pivoting.

<table border="1" class="dataframe">
  <thead>
    <tr>
      <th></th>
      <th>Month</th>
      <th>February</th>
      <th>January</th>
      <th>July</th>
      <th>March</th>
    </tr>
    <tr>
      <th>Name</th>
      <th>Product</th>
      <th></th>
      <th></th>
      <th></th>
      <th></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th rowspan="3" valign="top">Jones</th>
      <th>book</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
    <tr>
      <th>pen</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
    <tr>
      <th>ruler</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
    <tr>
      <th rowspan="3" valign="top">Smith</th>
      <th>book</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
    <tr>
      <th>pen</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
    <tr>
      <th>ruler</th>
      <td>0</td>
      <td>0</td>
      <td>0</td>
      <td>0</td>
    </tr>
  </tbody>
</table>

In [69]:
def average_seller(sales):
    return sales.groupby('Name')[['Total']].mean().rename(columns={'Total' : 'Average Sales'})

def product_name(sales):
    return sales.pivot_table(index='Name', columns='Product', values='Total', aggfunc='sum')

def count_product(sales):
    return sales.pivot_table(index=['Product', 'Name'], columns='Date', values='Total', aggfunc='count').fillna(0)

def total_by_month(sales):
    sales = sales.copy()

    sales['Month'] = sales['Date'].apply(
        lambda x : 
        'January' if x.split('.')[0] == '01' else
        'February' if x.split('.')[0] == '02' else
        'March' if x.split('.')[0] == '03' else
        'April' if x.split('.')[0] == '04' else
        'May' if x.split('.')[0] == '05' else
        'June' if x.split('.')[0] == '06' else
        'July' if x.split('.')[0] == '07' else
        'August' if x.split('.')[0] == '08' else
        'September' if x.split('.')[0] == '09' else
        'October' if x.split('.')[0] == '10' else
        'November' if x.split('.')[0] == '11' else
        'December'
    )

    return sales.pivot_table(index=['Name', 'Product'], columns='Month', values='Total', aggfunc='sum').fillna(0)


In [70]:
fp = Path('data') / 'sales.csv'
sales = pd.read_csv(fp)

sales

,Name,Product,Date,Total
0,Smith,book,01.01.2012,200
1,Jones,pen,02.20.2013,300
2,Trump,hotel,03.03.2015,100
3,Smith,book,05.10.2013,2000
4,Jones,book,02.20.2013,30
5,Trump,boat,03.30.2017,700
6,Smith,ruler,07.05.2014,2100
7,Jones,ruler,02.25.2015,350
8,Trump,book,03.03.2015,1000
9,Smith,pen,01.01.2012,2500


In [35]:
# don't change this cell -- it is needed for the tests to work
fp = Path('data') / 'sales.csv'
sales = pd.read_csv(fp)
q4_average_seller_out = average_seller(sales)
q4_product_name_out = product_name(sales)
q4_product_count_out = count_product(sales)
q4_total_by_month_out = total_by_month(sales)

In [36]:
grader.check("q4")

q4 results:
    q4 - 1 result:
        Trying:
            q4_average_seller_out.shape[0] == 3
        Expecting:
            True
        **********************************************************************
        Line 1, in q4 0
        Failed example:
            q4_average_seller_out.shape[0] == 3
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q4 0[0]>", line 1, in <module>
                q4_average_seller_out.shape[0] == 3
            AttributeError: 'NoneType' object has no attribute 'shape'

    q4 - 2 result:
        Trying:
            bool(q4_average_seller_out["Average Sales"].sum() < 5000)
        Expecting:
            True
        **********************************************************************
        Line 1, in q4 1
        Failed example:
            bool(q4_average_seller_out["Average Sales"].sum() < 5000)
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q4 1[0]>", line 1, in <module>
                bool(q4_average_seller_out["Average Sales"].sum() < 5000)
            TypeError: 'NoneType' object is not subscriptable

    q4 - 3 result:
        Trying:
            q4_product_name_out.size == 15
        Expecting:
            True
        **********************************************************************
        Line 1, in q4 2
        Failed example:
            q4_product_name_out.size == 15
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q4 2[0]>", line 1, in <module>
                q4_product_name_out.size == 15
            AttributeError: 'NoneType' object has no attribute 'size'

    q4 - 4 result:
        Trying:
            bool(q4_product_name_out["pen"].isnull().sum() == 0)
        Expecting:
            True
        **********************************************************************
        Line 1, in q4 3
        Failed example:
            bool(q4_product_name_out["pen"].isnull().sum() == 0)
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q4 3[0]>", line 1, in <module>
                bool(q4_product_name_out["pen"].isnull().sum() == 0)
            TypeError: 'NoneType' object is not subscriptable

    q4 - 5 result:
        Trying:
            q4_product_count_out.size == 70
        Expecting:
            True
        **********************************************************************
        Line 1, in q4 4
        Failed example:
            q4_product_count_out.size == 70
        Exception raised:
            Traceback (most recent call last):
              File "c:\Users\IdeRy\miniforge3\envs\dsc80\lib\doctest.py", line 1350, in __run
                exec(compile(example.source, filename, "single",
              File "<doctest q4 4[0]>", line 1, in <module>
                q4_product_count_out.size == 70
            AttributeError: 'NoneType' object has no attribute 'size'

    q4 - 6 result:
        Trying:
            q4_total_by_month_out.shape[1] == 5
        Expecting:
            True
        **********************************************************************
        Line 1, in q4 5
        Failed example:
            q4_total_by_month_out.shape[1] == 5
        Exception raised:
            Traceback (most recent call last):
 

## Congratulations! You're done Lab 3! 🏁

As a reminder, all of the work you want to submit needs to be in `lab.py`.

To ensure that all of the work you want to submit is in `lab.py`, we've included a script named `lab-validation.py` in the lab folder. You shouldn't edit it, but instead, you should call it from the command line (e.g. the Terminal) to test your work.

Once you've finished the lab, you should open the command line and run, in the directory for this lab:

```
python lab-validation.py
```

**This will run all of the `grader.check` cells that you see in this notebook, but only using the code in `lab.py` – that is, it doesn't look at any of the code in this notebook. If all of your `grader.check` cells pass in this notebook but not all of them pass in your command line with the above command, then you likely have code in your notebook that isn't in your `lab.py`!**

You can also use `lab-validation.py` to test individual questions. For instance,

```
python lab-validation.py q1 q2 q4
```

will run the `grader.check` cells for Questions 1, 2, and 4 – again, only using the code in `lab.py`. [This video](https://www.loom.com/share/0ea254b85b2745e59322b5e5a8692e91?sid=5acc92e6-0dfe-4555-9b6a-8115b6a52f99) how to use the script as well.

Once `python lab-validation.py` shows that you're passing all test cases, you're ready to submit your `lab.py` (and only your `lab.py`) to Gradescope. Once submitting to Gradescope, make sure to stick around until all test cases pass.

There is also a call to `grader.check_all()` below in _this_ notebook, but make sure to also follow the steps above.

<div class="alert alert-block alert-danger" markdown="1">

**There are only two functions in this lab in which you may use a `for`-loop:**
- **`read_linkedin_survey` in Question 1.**
- **`read_student_surveys` in Question 2.**
    
**You may lose points if you use a `for`-loop or `while`-loop in any other question!**

</div>

---

To double-check your work, the cell below will rerun all of the autograder tests.

In [37]:
grader.check_all()

q1 results: All test cases passed!

q2 results:
    q2 - 1 result:
        Test case passed!

    q2 - 2 result:
        Test case passed!

    q2 - 3 result:
        Test case passed!

    q2 - 4 result:
        Test case passed!

    q2 - 5 result:
        Trying:
            res = check_credit_out['ec'].max() == 6
        Expecting nothing
        ok
        Trying:
            bool(res)
        Expecting:
            True
        **********************************************************************
        Line 2, in q2 4
        Failed example:
            bool(res)
        Expected:
            True
        Got:
            False

q3 results:
    q3 - 1 result:
        Trying:
            isinstance(out_01, str)
        Expecting:
            True
        **********************************************************************
        Line 1, in q3 0
        Failed example:
            isinstance(out_01, str)
        Expected:
            True
        Got:
            False

    q